# Cirrhosis Survival Prediction - Final Notebook

This notebook is a clean, linear, reproducible pipeline for the Mohirdev/Kaggle multiclass competition.

## What this notebook does
- Loads data from Kaggle path or local `data/` path.
- Trains LightGBM, XGBoost, and CatBoost with stratified 5-fold CV.
- Reports CV log loss for each model.
- Finds a simple best blend on OOF predictions.
- Saves `submission.csv` with required columns.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_SPLITS = 5

In [ ]:
def resolve_data_paths():
    kaggle_base = '/kaggle/input/competitions/multiclassificationtask'
    local_base = 'data'

    if os.path.exists(os.path.join(kaggle_base, 'train.csv')):
        base = kaggle_base
    elif os.path.exists(os.path.join(local_base, 'train.csv')):
        base = local_base
    else:
        raise FileNotFoundError(
            "Could not find dataset. Put files in 'data/' locally or run on Kaggle."
        )

    return (
        os.path.join(base, 'train.csv'),
        os.path.join(base, 'test.csv'),
        os.path.join(base, 'sample_submission.csv'),
    )

train_path, test_path, sample_path = resolve_data_paths()
print('Train:', train_path)
print('Test :', test_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# Keep only official target classes for this competition.
train = train[train['Status'].isin(['C', 'CL', 'D'])].copy()

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('\nClass distribution:')
print(train['Status'].value_counts())

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Convert age from days to years.
    df['Age'] = df['Age'] / 365.25

    log_cols = ['Bilirubin', 'Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides']
    for col in log_cols:
        df[f'log_{col}'] = np.log1p(df[col])

    df['log_N_Days'] = np.log1p(df['N_Days'])
    df['Bili_Albumin_ratio'] = df['Bilirubin'] / (df['Albumin'] + 1e-6)
    df['SGOT_AlkPhos_ratio'] = df['SGOT'] / (df['Alk_Phos'] + 1e-6)
    df['Stage_Bili'] = df['Stage'] * df['Bilirubin']
    df['Proth_Bili'] = df['Prothrombin'] * df['Bilirubin']

    # Extra interaction features used in your experiments.
    df['Liver_Score_1'] = (df['Bilirubin'] * 0.5) + (df['Cholesterol'] * 0.003) - (df['Albumin'] * 0.5)
    df['Liver_Score_2'] = np.log1p(df['Copper']) + np.log1p(df['Alk_Phos']) + np.log1p(df['SGOT'])
    df['Platelets_Prothrombin'] = df['Platelets'] * df['Prothrombin']

    return df

train = engineer_features(train)
test = engineer_features(test)

cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
num_cols = [c for c in train.columns if c not in cat_cols + ['id', 'Status']]

le = LabelEncoder()
y = le.fit_transform(train['Status'])
X = train.drop(columns=['id', 'Status'])
X_test = test.drop(columns=['id'])

test_ids = test['id'].copy()

print('Classes:', list(le.classes_))
print('Features:', X.shape[1])

In [ ]:
def normalize_probs(probs):
    return probs / probs.sum(axis=1, keepdims=True)


def run_lgb_cv(X, y, X_test, cat_cols, num_cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(X), 3))
    test_pred = np.zeros((len(X_test), 3))

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
            ]), cat_cols),
        ]
    )

    params = dict(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        verbose=-1,
    )

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        X_tr_proc = preprocessor.fit_transform(X_tr)
        X_va_proc = preprocessor.transform(X_va)
        X_test_proc = preprocessor.transform(X_test)

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr_proc,
            y_tr,
            eval_set=[(X_va_proc, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )

        oof[va_idx] = model.predict_proba(X_va_proc)
        test_pred += model.predict_proba(X_test_proc) / n_splits
        print(f'LGB fold {fold} log_loss: {log_loss(y_va, oof[va_idx]):.5f}')

    return normalize_probs(oof), normalize_probs(test_pred)


def run_xgb_cv(X, y, X_test, cat_cols, num_cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(X), 3))
    test_pred = np.zeros((len(X_test), 3))

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
            ]), cat_cols),
        ]
    )

    class_counts = np.bincount(y)
    sample_weights = np.array([1.0 / np.sqrt(class_counts[yi]) for yi in y])
    sample_weights = sample_weights / sample_weights.mean()

    params = dict(
        objective='multi:softprob',
        num_class=3,
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        eval_metric='mlogloss',
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        verbosity=0,
    )

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        w_tr = sample_weights[tr_idx]

        X_tr_proc = preprocessor.fit_transform(X_tr)
        X_va_proc = preprocessor.transform(X_va)
        X_test_proc = preprocessor.transform(X_test)

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr_proc,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va_proc, y_va)],
            verbose=False,
        )

        oof[va_idx] = model.predict_proba(X_va_proc)
        test_pred += model.predict_proba(X_test_proc) / n_splits
        print(f'XGB fold {fold} log_loss: {log_loss(y_va, oof[va_idx]):.5f}')

    return normalize_probs(oof), normalize_probs(test_pred)


def run_cat_cv(X, y, X_test, cat_cols, num_cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(X), 3))
    test_pred = np.zeros((len(X_test), 3))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        num_imputer = SimpleImputer(strategy='median')
        cat_imputer = SimpleImputer(strategy='most_frequent')

        X_tr_num = X_tr[num_cols].copy()
        X_va_num = X_va[num_cols].copy()
        X_te_num = X_test[num_cols].copy()

        X_tr_num[:] = num_imputer.fit_transform(X_tr_num)
        X_va_num[:] = num_imputer.transform(X_va_num)
        X_te_num[:] = num_imputer.transform(X_te_num)

        X_tr_cat = X_tr[cat_cols].copy()
        X_va_cat = X_va[cat_cols].copy()
        X_te_cat = X_test[cat_cols].copy()

        X_tr_cat[:] = cat_imputer.fit_transform(X_tr_cat)
        X_va_cat[:] = cat_imputer.transform(X_va_cat)
        X_te_cat[:] = cat_imputer.transform(X_te_cat)

        X_tr_full = pd.concat([X_tr_num.reset_index(drop=True), X_tr_cat.reset_index(drop=True)], axis=1)
        X_va_full = pd.concat([X_va_num.reset_index(drop=True), X_va_cat.reset_index(drop=True)], axis=1)
        X_te_full = pd.concat([X_te_num.reset_index(drop=True), X_te_cat.reset_index(drop=True)], axis=1)

        cat_indices = [X_tr_full.columns.get_loc(c) for c in cat_cols]

        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            loss_function='MultiClass',
            eval_metric='MultiClass',
            random_seed=RANDOM_STATE,
            verbose=False,
            early_stopping_rounds=50,
        )

        model.fit(X_tr_full, y_tr, cat_features=cat_indices, eval_set=(X_va_full, y_va), verbose=False)

        oof[va_idx] = model.predict_proba(X_va_full)
        test_pred += model.predict_proba(X_te_full) / n_splits
        print(f'CAT fold {fold} log_loss: {log_loss(y_va, oof[va_idx]):.5f}')

    return normalize_probs(oof), normalize_probs(test_pred)

In [ ]:
lgb_oof, lgb_test = run_lgb_cv(X, y, X_test, cat_cols, num_cols, n_splits=N_SPLITS)
print('\nLGB CV log_loss:', round(log_loss(y, lgb_oof), 5))

xgb_oof, xgb_test = run_xgb_cv(X, y, X_test, cat_cols, num_cols, n_splits=N_SPLITS)
print('XGB CV log_loss:', round(log_loss(y, xgb_oof), 5))

cat_oof, cat_test = run_cat_cv(X, y, X_test, cat_cols, num_cols, n_splits=N_SPLITS)
print('CAT CV log_loss:', round(log_loss(y, cat_oof), 5))

In [ ]:
# Simple blend search on OOF predictions.
blend_candidates = [
    (0.3, 0.6, 0.1),
    (0.4, 0.5, 0.1),
    (0.4, 0.6, 0.0),
]

best_loss = 10.0
best_weights = None
best_test_pred = None

for xgb_w, cat_w, lgb_w in blend_candidates:
    oof_blend = normalize_probs(xgb_w * xgb_oof + cat_w * cat_oof + lgb_w * lgb_oof)
    loss = log_loss(y, oof_blend)
    print(f'XGB {xgb_w:.1f} | CAT {cat_w:.1f} | LGB {lgb_w:.1f} -> {loss:.5f}')

    if loss < best_loss:
        best_loss = loss
        best_weights = (xgb_w, cat_w, lgb_w)
        best_test_pred = normalize_probs(xgb_w * xgb_test + cat_w * cat_test + lgb_w * lgb_test)

print('\nBest blend weights:', best_weights)
print('Best OOF log_loss:', round(best_loss, 5))

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,
    'Status_C': best_test_pred[:, 0],
    'Status_CL': best_test_pred[:, 1],
    'Status_D': best_test_pred[:, 2],
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
print(submission.head())
print('Row probability sums:', submission[['Status_C', 'Status_CL', 'Status_D']].sum(axis=1).head().tolist())